# Exercise 2 — One-Hot Encoding, Bag of Words and N-Grams

Build a shared coordinate system, count tokens manually, inspect local order, then verify against a library. Complete reference cores 2.1–2.4 are in src; this notebook does not duplicate algorithms.

In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.preprocessing import lowercase_text, remove_punctuation, tokenize_words, calculate_word_frequencies
from src.classical_nlp import build_vocabulary, create_vocabulary_index, one_hot_encode, bag_of_words, build_document_term_matrix, generate_ngrams, countvectorizer_matrix

## 1. Load small workshop documents
Inspect this corpus before using larger academic documents.

In [ ]:
data = json.loads((ROOT / "data/examples/bow_examples.json").read_text(encoding="utf-8"))
texts = data["collections"][0]["texts"]
texts

## 2. Reuse Exercise 1
Lowercase, replace punctuation and tokenize. Retain stopwords and original word forms to isolate count representation.

In [ ]:
documents = [tokenize_words(remove_punctuation(lowercase_text(text))) for text in texts]
documents

## 3. Shared vocabulary — boundary 2.1
One sorted corpus vocabulary defines every document coordinate.

In [ ]:
vocabulary = build_vocabulary(documents)
vocabulary

## 4. Inspect the coordinate mapping

In [ ]:
index = create_vocabulary_index(vocabulary)
index

## 5. One-hot — boundary 2.2
One active coordinate, vocabulary-sized vector. Unknown terms raise ValueError.

In [ ]:
one_hot_encode("students", vocabulary)

## 6. Manual BoW — boundary 2.3
Repeated tokens increment counts in the same coordinate mapping.

In [ ]:
vectors = [bag_of_words(tokens, vocabulary) for tokens in documents]
vectors

## 7. Document-term matrix
Rows are documents; columns follow vocabulary. Matrix assembly reuses BoW.

In [ ]:
matrix = build_document_term_matrix(documents, vocabulary)
assert all(sum(row) == len(tokens) for row, tokens in zip(matrix, documents))
list(zip(texts, matrix))

## 8. Local order — boundary 2.4
Slide contiguous windows; do not cross document boundaries.

In [ ]:
[(n, generate_ngrams(documents[0], n)) for n in (1, 2, 3)]

## 9. Library verification, after manual understanding
CountVectorizer receives identical token lists through analyzer=list and the same fixed vocabulary. No further lowercasing or token-pattern filtering occurs. A callable analyzer bypasses ngram_range. Default raw-text settings can drop single-character tokens; do not compare differently processed inputs.

In [ ]:
library_matrix = countvectorizer_matrix(documents, vocabulary)
assert matrix == library_matrix
library_matrix

## 10. Limitations
Predict whether order changes the unigram vector, then inspect bigrams.

In [ ]:
order_documents = [tokenize_words(remove_punctuation(lowercase_text(text))) for text in ["Dog bites man.", "Man bites dog."]]
order_vocabulary = build_vocabulary(order_documents)
order_matrix = build_document_term_matrix(order_documents, order_vocabulary)
assert order_matrix[0] == order_matrix[1]
[generate_ngrams(tokens, 2) for tokens in order_documents]

Car and automobile have independent coordinates; semantic relatedness is not encoded. Common terms can dominate raw counts.

In [ ]:
one_hot_encode("car", ["automobile", "car"]), one_hot_encode("automobile", ["automobile", "car"])

In [ ]:
common = [["the", "the", "the", "language"], ["the", "data"]]
common_vocabulary = build_vocabulary(common)
common_vocabulary, build_document_term_matrix(common, common_vocabulary)

## 11. Mini challenge
D1: Students learn natural language processing

D2: Students learn machine learning

D3: Machine learning analyzes data

Before running: predict vocabulary size, students one-hot vector, each BoW vector, most repeated terms and D1 bigrams. Then verify with shared functions.

In [ ]:
challenge_texts = ["Students learn natural language processing", "Students learn machine learning", "Machine learning analyzes data"]
challenge = [tokenize_words(lowercase_text(text)) for text in challenge_texts]
challenge_vocabulary = build_vocabulary(challenge)
print("Vocabulary", len(challenge_vocabulary), challenge_vocabulary)
print("students", one_hot_encode("students", challenge_vocabulary))
print("BoW", build_document_term_matrix(challenge, challenge_vocabulary))
print("Counts", calculate_word_frequencies([token for tokens in challenge for token in tokens]))
print("D1 bigrams", generate_ngrams(challenge[0], 2))

## Checkpoint 2
Text can be represented numerically. Run `python -m pytest tests/test_classical_nlp.py`.

Bag of Words tells us how often words occur, but frequency alone does not tell us how informative a word is. Next we will build TF-IDF from scratch.